# Проект 1. Маркетинговый бюджет ПРАЙМ: куда он уходит и что возвращает

**Ситуация.** За 20 месяцев (январь 2025 - август 2026) ПРАЙМ потратил на привлечение
подписчиков около 196 млн ₽. Маркетинг просит увеличить бюджет performance-канала
еще на 30 % - это и так самый дорогой канал. Финансы сомневаются и просят проверить,
что эти деньги возвращаются

**Бизнес-вопрос.** Окупаются ли каналы привлечения, в которые уходит основная часть
бюджета и что ответить на заявку маркетинга?

**Гипотеза.** Каналы различаются не только ценой привлечения, но и качеством клиентов:
у дорогих каналов (performance, social) подписчики живут меньше, поэтому их
пожизненная выручка может не покрывать стоимость привлечения

**План.**

1. Зерно и правила расчета - с проверками
2. Контекст: как росла выручка
3. Факторная декомпозиция прироста выручки H1-2025 → H1-2026: сначала «где»
   (по каналам), потом «за счет чего» (база × частота × цена)
4. Качество каналов: удержание когорт
5. Юнит-экономика: CAC, LTV за 12 месяцев, окупаемость
6. Вывод и рекомендация

---

## 1. Подключение

Как на семинарах: строка подключения в `.env`, `q(sql)` открывает соединение
на запрос и сразу закрывает. Сутки и месяцы считаем по UTC - так они одинаковые у всех

In [ ]:
import math

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import text

from prime.config import get_engine

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Сутки и месяцы считаем по UTC - так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

---

## 2. Зерно расчета и правила

**Одна строка - одна подписка владельца.** Это не равно клиенту или платежу:

- Вопрос - про привлечение и окупаемость. Привлекается владелец подписки, он и платит, и отменяет. У владельца ровно одна подписка (проверим ниже),
  поэтому «подписка» и «платящий клиент» здесь совпадают
- **Не клиент**, потому что клиент может быть участником чужой подписки (шеринг).
  В `subscription_members` полмиллиона строк: платеж один на подписку, а членов
  несколько. Считать выручку «по клиентам» через членство задвоит деньги
  Платежи в базе привязаны к `subscription_id`, а не к клиенту, поэтому при таком
  зерне двойного счета нет по построению
- **Не платеж** - слишком мелко: вопрос про срок жизни и возврат денег, а платеж
  ничего не знает про удержание

Остальные правила:

- Выручка - только платежи со `status = 'success'` (`failed` не принесли денег,
  `refunded` вернули)
- Канал - `acquisition_channel` **владельца** подписки. Клиенты без канала
  показываем отдельной строкой «без канала», не выбрасываем и не прячем, чтобы части сошлись с итогом
- Период декомпозиции - **первое полугодие 2025 против первого полугодия 2026**:
  одинаковая длина (по 181 дню - поправка на длину месяцев не нужна), год к году -
  нет сезонности, а между ними случилось подорожание тарифов 01.03.2026

In [ ]:
# Проверка зерна: одна подписка на владельца.
subs_per_owner = q("""
    select max(n) as max_subs
    from (select owner_client_id, count(*) as n from prime.subscriptions group by 1) t
""")
assert subs_per_owner["max_subs"].iloc[0] == 1, "есть владельцы с несколькими подписками"
print("подписок на владельца: не больше одной - зерно «подписка» совпадает с «платящий клиент»")

# Масштаб шеринга: членов больше, чем подписок. Считать деньги по членам - задвоить.
sharing = q("""
    select (select count(*) from prime.subscription_members) as members,
           (select count(*) from prime.subscriptions) as subs
""")
print(f"участников подписок: {sharing['members'].iloc[0]:,}, подписок: {sharing['subs'].iloc[0]:,}"
      .replace(",", " "))

# Статусы платежей: что включаем в выручку, что нет.
statuses = q("select status, count(*) as n from prime.payments group by 1 order by 2 desc")
statuses["share_pct"] = (statuses["n"] / statuses["n"].sum() * 100).round(1)
print()
print(statuses.to_string(index=False))

---

## 3. Витрина «месяц × канал × тариф»

Витрина как на шестом семинаре, но на весь период данных: январь 2025 - июль 2026
(август 2026 неполный и с артефактом массовых окончаний подписок, поэтому его не берем).
Подписка **активна в месяце**, если началась до его конца и не закончилась до его
начала. Строка витрины - месяц × канал × тариф: месяц - атом планирования бюджета,
канал - то, о чем решение, тариф - для контроля структуры

In [ ]:
monthly = q("""
    with seg as (
        select client_id, coalesce(acquisition_channel, 'без канала') as channel
        from prime.clients
    ),
    act as (
        select to_char(m.month_start, 'YYYY-MM') as month, s.tariff_id, c.channel, count(*) as active
        from generate_series('2025-01-01'::timestamptz, '2026-07-01'::timestamptz, interval '1 month') as m(month_start)
        join prime.subscriptions s
          on s.started_at < m.month_start + interval '1 month'
         and (s.ended_at is null or s.ended_at >= m.month_start)
        join seg c on c.client_id = s.owner_client_id
        group by 1, 2, 3
    ),
    pay as (
        select to_char(date_trunc('month', p.paid_at), 'YYYY-MM') as month, s.tariff_id, c.channel,
               count(*) as n_payments, sum(p.amount) as revenue
        from prime.payments p
        join prime.subscriptions s on s.subscription_id = p.subscription_id
        join seg c on c.client_id = s.owner_client_id
        where p.status = 'success' and p.paid_at >= '2025-01-01' and p.paid_at < '2026-08-01'
        group by 1, 2, 3
    )
    select a.month, a.tariff_id, a.channel, a.active,
           coalesce(p.n_payments, 0) as n_payments, coalesce(p.revenue, 0) as revenue
    from act a left join pay p using (month, tariff_id, channel)
""")

assert monthly["month"].nunique() == 19, "витрина должна покрывать 19 месяцев"
assert monthly["revenue"].notna().all(), "потерялись строки платежей"

monthly.head(8)

---

## 4. Контекст: выручка растет, но «пилой»

Помесячная выручка пилит из-за разной длины месяцев (списание - раз в 30 дней,
это разбирали на шестом семинаре), поэтому смотрим на выручку **в день**.
Вертикальная линия - 01.03.2026, подорожание обоих тарифов

In [ ]:
m = monthly.groupby("month")[["revenue", "n_payments", "active"]].sum()
m["days"] = pd.PeriodIndex(m.index, freq="M").days_in_month
m["revenue_per_day"] = m["revenue"] / m["days"]

fig, ax = plt.subplots(figsize=(10, 4.2), layout="constrained")
ax.plot(m.index, m["revenue_per_day"] / 1e6, marker="o", color="#111827", zorder=2)
ax.axvline(14, color="#C1272D", linestyle="--", linewidth=1.2, zorder=1)
ax.text(14.2, m["revenue_per_day"].min() / 1e6, "01.03.2026\nподорожание тарифов",
        color="#C1272D", fontsize=9, va="bottom")
ax.set_title("Выручка в день растет все 19 месяцев; ступень - март 2026, подорожание", loc="left")
ax.set_ylabel("млн ₽ в день")
ax.set_xlabel("Месяц (по UTC)")
ax.tick_params(axis="x", rotation=45)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

print(f"январь 2025: {m['revenue_per_day'].iloc[0]/1e6:.2f} млн ₽/день → "
      f"июль 2026: {m['revenue_per_day'].iloc[-1]/1e6:.2f} млн ₽/день")

---

## 5. Декомпозиция. Шаг 1 - «где»: аддитивная модель по каналам

Выручка - сумма выручек каналов, значит ее изменение это сумма изменений каналов.
Сравниваем H1-2025 и H1-2026 (оба по 181 дню). Сумма вкладов обязана сойтись
с изменением итога до копейки - на этом стоит `assert`

In [ ]:
H1_A = [f"2025-0{i}" for i in range(1, 7)]
H1_B = [f"2026-0{i}" for i in range(1, 7)]

half = monthly.assign(half=monthly["month"].isin(H1_A).map({True: "H1-2025", False: None}))
half.loc[half["month"].isin(H1_B), "half"] = "H1-2026"

by_channel = (half[half["half"].notna()]
              .pivot_table(index="channel", columns="half", values="revenue", aggfunc="sum"))
by_channel["delta"] = by_channel["H1-2026"] - by_channel["H1-2025"]
by_channel["share_of_growth_pct"] = by_channel["delta"] / by_channel["delta"].sum() * 100
by_channel = by_channel.sort_values("delta", ascending=False)

total_A = monthly[monthly["month"].isin(H1_A)]["revenue"].sum()
total_B = monthly[monthly["month"].isin(H1_B)]["revenue"].sum()
assert abs(by_channel["delta"].sum() - (total_B - total_A)) < 0.01, "каналы не сходятся с итогом"

summary = by_channel.copy()
summary.loc["итого"] = summary.sum()
(summary / 1e6).assign(share_of_growth_pct=summary["share_of_growth_pct"]).round(
    {"H1-2025": 1, "H1-2026": 1, "delta": 1, "share_of_growth_pct": 1})

Таблица в млн ₽. Растут **все** каналы - одного виноватого или одного героя нет.
Но вклад в прирост распределен иначе, чем бюджет: organic дает ~31 % прироста,
performance ~17 %. А бюджета у них наоборот: у organic ~5 %, у performance ~50 %
(покажем в разделе 8). Первое подозрение на неэффективность

Чтобы понять, за счет чего растет выручка в целом, складывать недостаточно -
нужно раскладывать на множители

---

## 6. Декомпозиция. Шаг 2 - «за счет чего»: мультипликативная модель

> выручка = активные подписки × платежей на подписку × цена платежа

Активные подписки за полугодие - **среднее** по месяцам (суммировать базу за полгода
нельзя - одна подписка посчитается шесть раз). Порядок подстановок: сначала
количество, потом частота платежей, потом цена

In [ ]:
def contrib(x0: dict, x1: dict, order: list) -> dict:
    """Влияние каждого фактора: меняем их по одному в порядке order (цепные подстановки)."""
    cur, out = dict(x0), {}
    for f in order:
        before = math.prod(cur.values())
        cur[f] = x1[f]
        out[f] = math.prod(cur.values()) - before
    total = math.prod(x1.values()) - math.prod(x0.values())
    assert abs(sum(out.values()) - total) < 0.01, "сумма влияний не равна изменению"
    return out


def factors(revenue, n_payments, active) -> dict:
    return {"active": active, "per_sub": n_payments / active, "price": revenue / n_payments}


act_A = monthly[monthly["month"].isin(H1_A)].groupby("month")["active"].sum().mean()
act_B = monthly[monthly["month"].isin(H1_B)].groupby("month")["active"].sum().mean()
pay_A = monthly[monthly["month"].isin(H1_A)][["revenue", "n_payments"]].sum()
pay_B = monthly[monthly["month"].isin(H1_B)][["revenue", "n_payments"]].sum()

x0 = factors(pay_A["revenue"], pay_A["n_payments"], act_A)
x1 = factors(pay_B["revenue"], pay_B["n_payments"], act_B)

effects = pd.Series(contrib(x0, x1, ["active", "per_sub", "price"]))
total_delta = pay_B["revenue"] - pay_A["revenue"]
assert abs(effects.sum() - total_delta) < 0.01, "сумма влияний не равна изменению выручки"

factors_table = pd.DataFrame({"H1-2025": x0, "H1-2026": x1},
                             index=["active", "per_sub", "price"])
print("факторы:")
print(factors_table.round({"H1-2025": 2, "H1-2026": 2}).rename(
    index={"active": "активные подписки (ср. за месяц)",
           "per_sub": "платежей на подписку за полгода",
           "price": "цена платежа, ₽"}))
print()
print("влияния, млн ₽:")
print((effects / 1e6).round(1).rename(
    {"active": "активные подписки", "per_sub": "частота платежей", "price": "цена платежа"}))
print(f"сумма влияний {effects.sum()/1e6:.1f} млн ₽ = изменение выручки {total_delta/1e6:.1f} млн ₽")

In [ ]:
def waterfall(deltas: pd.Series, title: str) -> None:
    """Каскад изменений: по столбику на фактор и «итого» в конце."""
    before = deltas.cumsum() - deltas
    colors = ["#C1272D" if v < 0 else "#2A9D8F" for v in deltas]

    fig, ax = plt.subplots(figsize=(9, 4.4), layout="constrained")
    bars = ax.bar(deltas.index, deltas, bottom=before, color=colors, zorder=2)
    total = ax.bar("итого", deltas.sum(), color="#6B7280", zorder=2)
    for group in (bars, total):
        ax.bar_label(group, fmt=lambda v: f"{v:+,.1f}".replace(",", " ").replace("-", "−"), padding=3)
    ax.axhline(0, color="#111827", linewidth=0.8)
    ax.set_title(title, loc="left")
    ax.set_ylabel("млн ₽")
    ax.set_xlabel("H1-2025 → H1-2026. Порядок подстановок: база → частота платежей → цена.")
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()


wf = (effects / 1e6).rename({"active": "активные\nподписки",
                             "per_sub": "частота\nплатежей",
                             "price": "цена\nплатежа"})
waterfall(wf, "Прирост выручки +138 млн ₽ за год: 70 % дала база, 26 % - цена")

**Вывод по декомпозиции.** Прирост +138,3 млн ₽ (+75 % год к году): база подписок
дала +96,7 млн (70 %), подорожание - +35,9 млн (26 %), частота платежей - +5,7 млн (4 %).
Главный двигатель роста - не цена, а прирост базы. Но база у каналов общая по затратам
и разная по качеству: проверяем, какие каналы приводят подписки, которые живут

---

## 7. Качество каналов: удержание когорт

Берем когорты подписок, начавшихся в H1-2025, и следим за ними 12 месяцев жизни -
данные позволяют (последнее наблюдение - 30.08.2026, у июньской когорты 12-й месяц
заканчивается в июне 2026). Зерно - подписка × месяц жизни. Метрика - доля подписок
когорты, доживших до месяца `k`

In [ ]:
retention = q("""
    with s as (
        select s.subscription_id,
               coalesce(c.acquisition_channel, 'без канала') as channel,
               s.started_at,
               coalesce(s.ended_at, '2026-09-01'::timestamptz) as end_eff
        from prime.subscriptions s
        join prime.clients c on c.client_id = s.owner_client_id
        where s.started_at >= '2025-01-01' and s.started_at < '2025-07-01'
    )
    select s.channel, k.n as month_of_life,
           avg((s.end_eff >= s.started_at + make_interval(months => k.n))::int) as alive
    from s cross join generate_series(1, 12) as k(n)
    group by 1, 2
""")

cohort_size = q("""
    select coalesce(c.acquisition_channel, 'без канала') as channel, count(*) as subs
    from prime.subscriptions s
    join prime.clients c on c.client_id = s.owner_client_id
    where s.started_at >= '2025-01-01' and s.started_at < '2025-07-01'
    group by 1 order by 2 desc
""")
print("размеры когорт H1-2025:")
print(cohort_size.to_string(index=False))

surv = retention.pivot_table(index="month_of_life", columns="channel", values="alive") * 100
surv.round(1)

In [ ]:
focus = ["performance", "social"]
fig, ax = plt.subplots(figsize=(9.5, 4.6), layout="constrained")
for ch in surv.columns:
    if ch in focus:
        ax.plot(surv.index, surv[ch], color="#C1272D", linewidth=2.2,
                linestyle="-" if ch == "performance" else "--", label=ch, zorder=3)
    elif ch == "organic":
        ax.plot(surv.index, surv[ch], color="#111827", linewidth=2.2, label=ch, zorder=3)
    else:
        ax.plot(surv.index, surv[ch], color="#9CA3AF", linewidth=1.2, label=ch, zorder=2)
ax.set_title("Через 12 месяцев живы 76 % подписок из органики - и только 42–51 % из social и performance",
             loc="left")
ax.set_xlabel("Месяц жизни подписки")
ax.set_ylabel("доля живых подписок когорты, %")
ax.set_xticks(range(1, 13))
ax.legend(frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

r12 = surv.loc[12].sort_values()
print("удержание 12-го месяца, %:")
print(r12.round(1).to_string())

Разрыв огромный: к году жизни у social остается ~42 % подписок против ~76 %
у organic. Возможно у «плохих» каналов просто чек меньше и дело не в удержании.
Нужно проверить среднюю выручку на активную подписку по каналам (окно H1-2025, чтобы
не мешать с подорожанием)

In [ ]:
arpu = (monthly[monthly["month"].isin(H1_A)]
        .groupby("channel")[["revenue", "active"]].sum())
arpu["arpu_month"] = arpu["revenue"] / arpu["active"]
arpu = arpu.sort_values("arpu_month", ascending=False)

print("выручка на активную подписку в месяц, H1-2025, ₽:")
print(arpu["arpu_month"].round(0).to_string())

spread = arpu["arpu_month"].max() / arpu["arpu_month"].min() - 1
print(f"\nразброс между каналами: {spread*100:.0f} % - при разнице удержания почти в 2 раза")

Чек у каналов одинаковый с точностью до ~1 % (тарифная структура везде одна -
~62 % «Базового»). Значит, LTV каналов различается **длиной жизни**, а не деньгами
в месяц. Дорогие каналы приводят подписки, которые умирают раньше. Осталось
сравнить с ценой привлечения

---

## 8. Юнит-экономика: CAC и LTV за 12 месяцев

- **CAC** - расходы канала / новые подписки канала. Окно - январь–июль 2025:
  то же, что у когорт LTV, чтобы не сравнивать цену одного года с выручкой другого.
- **LTV₁₂** - средняя выручка подписки за первые 12 месяцев жизни. Когорты -
  старты января–июля 2025: за июльскими 12 месяцев заканчиваются 31.07.2026,
  наблюдаем их полностью. Усечение на 12 месяцах - компромисс: дальше когорты
  наблюдаются не целиком, а экстраполировать хвост - лишнее допущение.
- LTV считаем по **выручке**, не по марже (кэшбэк и себестоимость сервисов
  не вычтены). Вернемся к этому ограничению в выводе

In [ ]:
cac = q("""
    with seg as (
        select client_id, coalesce(acquisition_channel, 'без канала') as channel
        from prime.clients
    ),
    new_subs as (
        select c.channel, count(*) as new_subs
        from prime.subscriptions s
        join seg c on c.client_id = s.owner_client_id
        where s.started_at >= '2025-01-01' and s.started_at < '2025-08-01'
        group by 1
    ),
    spend_window as (
        select channel, sum(amount) as spend
        from prime.marketing_spend
        where spend_date >= '2025-01-01' and spend_date < '2025-08-01'
        group by 1
    ),
    spend_total as (
        select channel, sum(amount) as spend_total
        from prime.marketing_spend
        group by 1
    )
    select n.channel, n.new_subs, w.spend as spend_window, t.spend_total,
           w.spend / n.new_subs as cac
    from new_subs n
    left join spend_window w using (channel)
    left join spend_total t using (channel)
""")

ltv = q("""
    with s as (
        select s.subscription_id,
               coalesce(c.acquisition_channel, 'без канала') as channel,
               s.started_at
        from prime.subscriptions s
        join prime.clients c on c.client_id = s.owner_client_id
        where s.started_at >= '2025-01-01' and s.started_at < '2025-08-01'
    ),
    rev as (
        select s.subscription_id, sum(p.amount) as rev12
        from s
        join prime.payments p on p.subscription_id = s.subscription_id
         and p.status = 'success'
         and p.paid_at < s.started_at + interval '12 months'
        group by 1
    )
    select s.channel, count(*) as subs, avg(coalesce(r.rev12, 0)) as ltv12
    from s left join rev r using (subscription_id)
    group by 1
""")

unit = cac.merge(ltv, on="channel", validate="1:1")
unit["ltv_cac"] = unit["ltv12"] / unit["cac"]
unit["budget_share_pct"] = unit["spend_total"] / unit["spend_total"].sum() * 100
unit = unit.sort_values("cac", ascending=False, na_position="last")

show = unit[["channel", "new_subs", "cac", "ltv12", "ltv_cac", "budget_share_pct"]].copy()
show.columns = ["канал", "новых подписок за янв–июл 2025", "CAC, ₽",
                "LTV за 12 мес, ₽", "LTV/CAC", "доля бюджета, %"]
show.round({"CAC, ₽": 0, "LTV за 12 мес, ₽": 0, "LTV/CAC": 1, "доля бюджета, %": 1})

In [ ]:
paid = unit.dropna(subset=["cac"])

fig, ax = plt.subplots(figsize=(9.5, 4.4), layout="constrained")
x = range(len(paid))
w = 0.38
b1 = ax.bar([i - w/2 for i in x], paid["cac"], width=w, color="#C1272D", label="CAC - цена привлечения", zorder=2)
b2 = ax.bar([i + w/2 for i in x], paid["ltv12"], width=w, color="#2A9D8F",
            label="LTV - выручка за первые 12 мес", zorder=2)
ax.bar_label(b1, fmt=lambda v: f"{v:,.0f}".replace(",", " "), fontsize=8, padding=2)
ax.bar_label(b2, fmt=lambda v: f"{v:,.0f}".replace(",", " "), fontsize=8, padding=2)
ax.set_xticks(list(x), paid["channel"])
ax.set_title("Performance покупает подписку за ~2 100 ₽ - а она возвращает ~2 600 ₽ за год", loc="left")
ax.set_ylabel("₽ на подписку")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6), layout="constrained")
colors = ["#C1272D" if c in ("performance", "social") else "#9CA3AF" for c in paid["channel"]]
ax.scatter(paid["budget_share_pct"], paid["ltv_cac"], s=70, color=colors, zorder=3)
for _, row in paid.iterrows():
    ax.annotate(row["channel"], (row["budget_share_pct"], row["ltv_cac"]),
                textcoords="offset points", xytext=(8, 4), fontsize=9)
ax.axhline(1, color="#111827", linewidth=0.8, linestyle="--")
ax.text(0.5, 1.15, "LTV = CAC: ниже линии канал не окупается",
        ha="left", fontsize=8, color="#111827")
ax.set_xlim(-2, 58)
ax.set_title("Больше всего бюджета - у каналов с худшей окупаемостью", loc="left")
ax.set_xlabel("Доля в маркетинговом бюджете за весь период, %")
ax.set_ylabel("LTV за 12 мес / CAC, раз")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

**Performance окупается с запасом 1,25×** - и это по выручке, до вычета кэшбэка
и себестоимости сервисов. По марже канал, скорее всего, около нуля или в минусе.
При этом на него и social - каналы с худшей окупаемостью - уходит ~69 % бюджета,
а на referral и CRM с окупаемостью 6–10× - меньше 8 %

---

## 9. Вывод и рекомендации

**Что случилось.** За год (H1-2025 → H1-2026) выручка подписки выросла на
+138,3 млн ₽ (+75 %). Маркетинг просит +30 % к бюджету performance - канала,
на который уже уходит половина бюджета

**За счет чего.** Декомпозиция: +96,7 млн дал прирост базы, +35,9 млн - подорожание,
+5,7 млн - частота платежей. Растут все каналы, но качество базы разное:
удержание 12-го месяца - 76 % у organic против 51 % у performance и 42 % у social,
при почти одинаковом чеке. Поэтому LTV за год у performance - ~2 600 ₽ при CAC
~2 100 ₽ (окупаемость 1,25×), а у referral и CRM - 6–10× при доле бюджета меньше 8 %

**Чего мы не знаем.** (1) LTV посчитан по выручке: после вычета кэшбэка
и себестоимости сервисов performance, скорее всего, не окупается вовсе - расчет
по марже только ухудшит его позицию. (2) Эластичность каналов: не факт, что CAC
referral сохранится при увеличении бюджета в разы - масштабировать надо пилотом
с контролем CAC помесячно. (3) Атрибуция: часть «organic» - отложенный эффект
платных каналов, полный отказ от performance может просадить и органику

**Что делать.** Заявку на +30 % к performance - **отклонить** в текущем виде:
канал работает на грани окупаемости. Вместо этого:
1. performance - заморозить бюджет на уровне H1-2026, выключить кампании
   с CAC выше медианного по каналу;
2. +2–3 п.п. бюджета переложить в referral-программу пилотом на квартал
   с ежемесячным контролем CAC: если CAC вырастет выше ~700 ₽, пилот свернуть;
3. разобраться с ранним оттоком платных когорт: провал в удержании у social
   и performance начинается с первых трех месяцев - это зона ответственности
   онбординга, а не только закупки

Контрольная точка - через квартал: пересчитать LTV/CAC по когортам Q3-2026

---

**Что проверяли и отвергли.** (1) «Подорожание 01.03.2026 вызвало волну отмен» -
не подтверждается: отток по месяцам стабилен ~4 % и до, и после, всплеска
`too_expensive` нет (плюс тот же результат на сегменте в ДЗ-2). (2) «В ноябре 2025
рост остановился» - артефакт длины месяца: в пересчете на день рост не прерывался.
(3) «Дело в тарифной структуре каналов» - нет: доля «Базового» ~62 % во всех каналах,
разница LTV идет через удержание